# Knee MRI Classifier — MedGemma Fine-Tuning for ACL Tear Detection

**Course:** COMP 6630 Machine Learning, Spring 2026  
**Team:** Gabriella Hawkes, Elizabeth Casey, Maddie Larkin  
**Dataset:** MRNet (Stanford) — knee MRI exams labeled for ACL tear  
**Model:** google/medgemma-4b-it (vision-language model, pretrained on medical imagery)

This notebook establishes a zero-shot baseline and fine-tunes MedGemma on the MRNet dataset for binary ACL tear classification.

### How to use
1. Set `VIEWPOINT` (cell below) to one of `'sagittal'`, `'coronal'`, `'axial'`.
2. Run all cells top-to-bottom. The notebook saves results under filenames that include the viewpoint, so you can run all three back-to-back without overwriting.

### GenAI Disclosure
Used Claude and Gemini for: data pipeline debugging (parquet loading, fixed-shape decoder workarounds), out-of-memory diagnosis, model-loading boilerplate, baseline evaluation code, fine-tuning scaffold based on Google's official tutorial, and final report drafting.

## 1. Setup

In [ ]:
!pip install -q bitsandbytes accelerate peft trl

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from huggingface_hub import login, hf_hub_download, snapshot_download
from transformers import pipeline
from torchvision import transforms
from torch import Tensor
from sklearn.preprocessing import normalize
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import glob


### HuggingFace login

**Before running this cell:**
1. Accept the MedGemma license at https://huggingface.co/google/medgemma-4b-it.
2. Generate a **Read** access token at https://huggingface.co/settings/tokens (fine-grained tokens need extra config — Read is simpler).
3. Paste the token when prompted below.

In [ ]:
login()

## 2. Choose viewpoint

**Set `VIEWPOINT` here.** This single variable controls baseline, fine-tuning, and evaluation throughout the notebook. Output filenames automatically include the viewpoint name, so different runs don't overwrite each other.

In [ ]:
VIEWPOINT = "axial"   # one of: 'sagittal', 'coronal', 'axial'
assert VIEWPOINT in ("sagittal", "coronal", "axial"), VIEWPOINT

## 3. Download and inspect the dataset

In [ ]:
def download_one_example(repo_id, filepath):
    """Download one parquet shard from HF and return it as an HF Dataset."""
    filepath1 = hf_hub_download(
        repo_id=repo_id, filename=filepath, repo_type="dataset",
    )
    return Dataset(pq.read_table(filepath1))


In [ ]:
example = download_one_example(
    "AUMLProject/mrnet-acl",
    "data/train-00000-of-00029.parquet",
)
print(example)

### Load full dataset

We use MRNet's official validation split as the test set so our numbers are comparable to published benchmarks. `n_files` controls how much training data to load (None = all 29 shards).

In [ ]:
def download_train_test_datasets(seed=42, n_files=None):
    if n_files is None:
        train_ds = load_dataset("AUMLProject/mrnet-acl", split="train")
    else:
        data_files = [f"data/train-{i:05d}-of-00029.parquet" for i in range(n_files)]
        train_ds = load_dataset(
            "AUMLProject/mrnet-acl",
            data_files={"train": data_files},
            split="train",
            verification_mode="no_checks",
        )
    test_ds = load_dataset("AUMLProject/mrnet-acl", split="validation")
    ds = DatasetDict({"train": train_ds, "test": test_ds})
    print(f"Train size: {len(ds['train'])}")
    print(f"Test size:  {len(ds['test'])}")
    print(f"Train label dist: {pd.Series(ds['train']['label']).value_counts().to_dict()}")
    print(f"Test label dist:  {pd.Series(ds['test']['label']).value_counts().to_dict()}")
    return ds


In [ ]:
# Use n_files=3 for fast dev (~117 train examples), n_files=None for full set.
ds = download_train_test_datasets(n_files=3)

In [ ]:
print(ds)

### Inspect one sample to confirm variable slice counts

The dataset declares each view as fixed shape (32, 224, 224) but actual exams have variable slice counts. We bypass the buggy decoder by reading raw PyArrow.

In [ ]:
test_table = ds["test"].data.table
row_idx = 0
for view in ["sagittal", "coronal", "axial"]:
    raw = test_table.column(view)[row_idx].as_py()
    arr = np.array(raw)
    print(f"{view}: shape {arr.shape}, dtype {arr.dtype}")
label = test_table.column("label")[row_idx].as_py()
print(f"label: {label}")

## 4. Zero-shot baseline (no fine-tuning)

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from transformers import pipeline
from PIL import Image
import bitsandbytes as bnb
import accelerate


In [ ]:
# BUGFIX: keep at medgemma-4b-it. The 1.5 variant has loading issues with
# 4-bit quant via AutoModelForImageTextToText.
model_id = "google/medgemma-4b-it"

In [ ]:
model_kwargs = dict(
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
)
pipe = pipeline("image-text-to-text", model=model_id, model_kwargs=model_kwargs)

### Helper functions

In [ ]:
def get_sample(ds_split, idx):
    """Read one sample using PyArrow directly to bypass the fixed-shape decoder."""
    table = ds_split.data.table
    return {
        "sagittal": np.array(table.column("sagittal")[idx].as_py()),
        "coronal":  np.array(table.column("coronal")[idx].as_py()),
        "axial":    np.array(table.column("axial")[idx].as_py()),
        "label":    int(table.column("label")[idx].as_py()),
    }


def slice_to_pil(mri_stack, slice_idx=None):
    """Convert one slice of a 3D MRI stack to a PIL image, rescaled to [0, 255]."""
    n_slices = mri_stack.shape[0]
    if slice_idx is None:
        slice_idx = n_slices // 2
    if not (0 <= slice_idx < n_slices):
        raise ValueError(f"slice_idx {slice_idx} out of [0, {n_slices})")
    sl = mri_stack[slice_idx].astype(np.float32)
    sl_min, sl_max = sl.min(), sl.max()
    if sl_max - sl_min < 1e-8:
        rescaled = np.zeros_like(sl, dtype=np.uint8)
    else:
        rescaled = (255.0 * (sl - sl_min) / (sl_max - sl_min)).astype(np.uint8)
    return Image.fromarray(rescaled)


def parse_yes_no(response_text):
    """Parse MedGemma's free-text response into 0/1; None if unclear."""
    if not response_text:
        return None
    txt = response_text.strip().lower()
    words = txt.split()
    if not words:
        return None
    first = words[0].strip(".,:;!?'\"")
    if first == "yes": return 1
    if first == "no": return 0
    if "yes" in txt and "no" not in txt: return 1
    if "no" in txt and "yes" not in txt: return 0
    return None


### Inference function — no label leakage

`predict_acl` builds a user-only message. The model has to generate its own answer.

In [ ]:
def predict_acl(sample, pipe, view=None, num_center_slices=1):
    """Zero-shot or fine-tuned ACL tear prediction on one sample."""
    if view is None:
        view = VIEWPOINT  # BUGFIX: default to global VIEWPOINT, not 'sagittal'
    stack = sample[view]
    n_slices = stack.shape[0]
    mid = n_slices // 2
    num_each_side = num_center_slices // 2
    is_odd = 1 if num_center_slices % 2 != 0 else 0
    start = max(0, mid - num_each_side)
    end = min(n_slices, mid + num_each_side + is_odd)
    images = [slice_to_pil(stack, i) for i in range(start, end)]

    prompt = (
        "You are looking at a knee MRI slice. Does this image "
        "show evidence of an ACL (Anterior Cruciate Ligament) "
        "tear? Answer with a single word: 'Yes' or 'No'."
    )
    user_content = [{"type": "text", "text": prompt}]
    for img in images:
        user_content.append({"type": "image", "image": img})
    messages = [{"role": "user", "content": user_content}]

    with torch.no_grad():
        output = pipe(text=messages, max_new_tokens=10, do_sample=False)
        response = output[0]["generated_text"][-1]["content"]

    return {
        "response": response,
        "prediction": parse_yes_no(response),
        "label": sample["label"],
        "view": view,
    }


### Training-format helper (assistant turn included — for SFT only)

In [ ]:
def reformat_sample(stack, label, num_center_slices=1):
    """Build one chat-format training example (user + assistant). Includes
    the ground-truth label so SFTTrainer learns from it. NEVER use for inference."""
    n_slices = stack.shape[0]
    mid = n_slices // 2
    num_each_side = num_center_slices // 2
    is_odd = 1 if num_center_slices % 2 != 0 else 0
    start = max(0, mid - num_each_side)
    end = min(n_slices, mid + num_each_side + is_odd)
    images = [slice_to_pil(stack, i) for i in range(start, end)]

    prompt = (
        "You are looking at a knee MRI slice. Does this image "
        "show evidence of an ACL (Anterior Cruciate Ligament) "
        "tear? Answer with a single word: 'Yes' or 'No'."
    )
    user_content = [{"type": "text", "text": prompt}]
    for img in images:
        user_content.append({"type": "image", "image": img})
    answer = "Yes" if label == 1 else "No"
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": [{"type": "text", "text": answer}]},
    ]
    return {"messages": messages, "image": images[0]}


def reformat_ds_at_index(ds_split, idx, view, num_center_slices=1):
    table = ds_split.data.table
    stack = np.array(table.column(view)[idx].as_py())
    label = int(table.column("label")[idx].as_py())
    return reformat_sample(stack, label, num_center_slices)


def build_finetuning_dataset(ds_split, view, num_center_slices=1):
    """Build an HF Dataset with 'messages' and 'image' columns for SFTTrainer."""
    from datasets import Dataset as HFDataset
    records = [
        reformat_ds_at_index(ds_split, idx, view, num_center_slices)
        for idx in range(len(ds_split))
    ]
    return HFDataset.from_list(records)


### Sanity check (one sample)

In [ ]:
sample = get_sample(ds["test"], 0)
result = predict_acl(sample, pipe, view=VIEWPOINT)
print(f"Label:        {result['label']}")
print(f"Prediction:   {result['prediction']}")
print(f"Raw response: {result['response']!r}")

### Baseline evaluation

Full test set (n=120). Metrics: accuracy, precision, recall, F1, confusion matrix.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
from tqdm.auto import tqdm


def evaluate(pipe, test_ds, n_samples=None, view=None, seed=42,
             verbose=False, label="MedGemma"):
    if view is None:
        view = VIEWPOINT
    if n_samples is None:
        indices = np.arange(len(test_ds))
    else:
        n = min(n_samples, len(test_ds))
        indices = np.random.RandomState(seed).choice(len(test_ds), size=n, replace=False)

    results = []
    for idx in tqdm(indices, desc=f"Evaluating {label} ({view})"):
        sample = get_sample(test_ds, int(idx))
        try:
            r = predict_acl(sample, pipe, view=view)
            r["idx"] = int(idx)
            results.append(r)
            if verbose:
                print(f"[{idx}] label={r['label']} pred={r['prediction']} | "
                      f"{r['response'][:60]}")
        except Exception as e:
            print(f"[{idx}] ERROR: {e}")
            results.append({
                "idx": int(idx), "prediction": None,
                "label": sample["label"], "response": f"ERROR: {e}",
                "view": view,
            })

    y_true, y_pred = [], []
    n_unparsed = 0
    for r in results:
        y_true.append(int(r["label"]))
        if r["prediction"] is None:
            n_unparsed += 1
            y_pred.append(1 - int(r["label"]))
        else:
            y_pred.append(int(r["prediction"]))

    metrics = {
        "n_evaluated": len(y_true),
        "n_unparseable": n_unparsed,
        "view": view,
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }

    print("\n" + "=" * 60)
    print(f"{label} — view={view}, n={metrics['n_evaluated']}")
    print("=" * 60)
    print(f"Accuracy:  {metrics['accuracy']:.3f}")
    print(f"Precision: {metrics['precision']:.3f}")
    print(f"Recall:    {metrics['recall']:.3f}")
    print(f"F1:        {metrics['f1']:.3f}")
    print(f"Confusion: {metrics['confusion_matrix']}")
    print("  (rows = true [no-tear, tear], cols = predicted [no-tear, tear])")
    print(f"Unparseable responses: {metrics['n_unparseable']}")
    print()
    print(classification_report(
        y_true, y_pred, target_names=["no-tear", "tear"], zero_division=0
    ))
    return results, metrics


In [ ]:
# Baseline on the full test set, on the chosen viewpoint.
baseline_results, baseline_metrics = evaluate(
    pipe, ds["test"],
    n_samples=None,
    view=VIEWPOINT,
    label="Zero-shot MedGemma",
    verbose=False,
)

In [ ]:
import json

# BUGFIX: filename uses VIEWPOINT so different runs don't overwrite each other.
baseline_path = f"baseline_results_{VIEWPOINT}.json"
with open(baseline_path, "w") as f:
    json.dump({
        "model": model_id,
        "fine_tuned": False,
        "view": VIEWPOINT,
        "n_samples": baseline_metrics["n_evaluated"],
        "metrics": baseline_metrics,
    }, f, indent=2)

print(f"Saved {baseline_path}")
print(json.dumps(baseline_metrics, indent=2))

## 5. Fine-tune MedGemma with LoRA

LoRA adapters on top of the 4-bit quantized base. Following: https://github.com/google-health/medgemma/blob/main/notebooks/fine_tune_with_hugging_face.ipynb

In [ ]:
# Free the zero-shot pipeline before loading the training model.
import gc
try:
    del pipe
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

### Optional: mount Drive for checkpoint backup

If Colab disconnects mid-training, a 4-hour run is gone. Mounting Drive lets us save the LoRA adapters there at the end so the next session can reuse them.

Skip this cell if you don't want Drive integration.

In [ ]:
# Mount Drive for checkpoint backup. Skip this cell if you don't want it.
from google.colab import drive
drive.mount("/content/drive")
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/medgemma-mrnet"
import os
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"Drive backup dir: {DRIVE_BACKUP_DIR}")

### Build training-format datasets

In [ ]:
def load_train_eval_ds(view):
    train_ds = build_finetuning_dataset(ds["train"], view=view, num_center_slices=1)
    eval_ds  = build_finetuning_dataset(ds["test"],  view=view, num_center_slices=1)
    print(f"Train: {len(train_ds)} examples")
    print(f"Eval:  {len(eval_ds)} examples")
    print("\nFirst example keys:", list(train_ds[0].keys()))
    print("Sample messages (first example):")
    for msg in train_ds[0]["messages"]:
        print(f"  {msg['role']}: {[c.get('type') for c in msg['content']]}")
    return train_ds, eval_ds


### Load training model

In [ ]:
if torch.cuda.get_device_capability()[0] < 8:
    model_kwargs = dict(
        attn_implementation="eager",
        torch_dtype=torch.float16,
        device_map="auto",
    )
else:
    model_kwargs = dict(
        attn_implementation="eager",
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=model_kwargs["torch_dtype"],
    bnb_4bit_quant_storage=model_kwargs["torch_dtype"],
)
model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
auto_processor = AutoProcessor.from_pretrained(model_id)
auto_processor.tokenizer.padding_side = "right"

### LoRA config

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16, lora_dropout=0.05, r=16, bias="none",
    target_modules="all-linear", task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"],
)

### Collator

In [ ]:
from typing import Any


def collate_fn(examples: list[dict[str, Any]]):
    texts, images = [], []
    for example in examples:
        images.append([example["image"].convert("RGB")])
        texts.append(auto_processor.apply_chat_template(
            example["messages"], add_generation_prompt=False, tokenize=False,
        ).strip())
    batch = auto_processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    image_token_id = [
        auto_processor.tokenizer.convert_tokens_to_ids(
            auto_processor.tokenizer.special_tokens_map["boi_token"]
        )
    ]
    labels[labels == auto_processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100
    batch["labels"] = labels
    return batch

### Train function

In [ ]:
from trl import SFTConfig, SFTTrainer


def train(learning_rate, num_epochs, train_ds, eval_ds, view):
    """Fine-tune MedGemma on the given datasets and return the trainer.

    BUGFIX: was not returning trainer, so finetuned_pipe couldn't access
    trainer.model afterward. Also wires num_epochs through to SFTConfig.
    """
    sft_config = SFTConfig(
        output_dir=f"medgemma-mrnet-{view}",
        num_train_epochs=num_epochs,    # BUGFIX: actually use the param
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        per_device_eval_batch_size=2,
        learning_rate=learning_rate,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        optim="adamw_torch_fused",
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        bf16=(torch.cuda.get_device_capability()[0] >= 8),
        fp16=(torch.cuda.get_device_capability()[0] < 8),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataset_kwargs={"skip_prepare_dataset": True},
        remove_unused_columns=False,
        report_to="none",
    )
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collate_fn,
        peft_config=peft_config,
        processing_class=auto_processor,
    )
    trainer.train()

    # Save adapters locally
    local_dir = f"medgemma-mrnet-{learning_rate}-{view}/final"
    trainer.save_model(local_dir)
    print(f"Saved adapters to {local_dir}")

    # Optional: also copy to Drive if mounted (so a Colab disconnect doesn't kill us)
    try:
        import shutil, os
        if os.path.exists("/content/drive/MyDrive"):
            drive_dir = f"/content/drive/MyDrive/medgemma-mrnet/{view}-lr{learning_rate}"
            shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
            print(f"Backed up adapters to {drive_dir}")
    except Exception as e:
        print(f"(Drive backup skipped: {e})")

    return trainer    # BUGFIX: return so caller can use trainer.model


### Run training

In [ ]:
lr = 2e-4
num_epochs = 1
print(f"=== TRAINING START — view={VIEWPOINT}, lr={lr}, epochs={num_epochs} ===")
train_ds, eval_ds = load_train_eval_ds(VIEWPOINT)
trainer = train(lr, num_epochs, train_ds, eval_ds, VIEWPOINT)
print("=== TRAINING DONE ===")

## 6. Evaluate the fine-tuned model

Same `evaluate()` function as the baseline → numbers are directly comparable.

In [ ]:
# BUGFIX: pipeline() takes tokenizer + image_processor (not processor=).
finetuned_pipe = pipeline(
    "image-text-to-text",
    model=trainer.model,
    tokenizer=auto_processor.tokenizer,
    image_processor=auto_processor.image_processor,
)

In [ ]:
sample = get_sample(ds["test"], 0)
result = predict_acl(sample, finetuned_pipe, view=VIEWPOINT)
print(f"Label:        {result['label']}")
print(f"Prediction:   {result['prediction']}")
print(f"Raw response: {result['response']!r}")

In [ ]:
finetuned_results, finetuned_metrics = evaluate(
    finetuned_pipe, ds["test"],
    n_samples=None,
    view=VIEWPOINT,
    label="Fine-tuned MedGemma",
    verbose=False,
)

In [ ]:
finetuned_path = f"finetuned_results_{VIEWPOINT}.json"
with open(finetuned_path, "w") as f:
    json.dump({
        "model": model_id,
        "fine_tuned": True,
        "view": VIEWPOINT,
        "n_samples": finetuned_metrics["n_evaluated"],
        "metrics": finetuned_metrics,
    }, f, indent=2)
print(f"Saved {finetuned_path}")

## 7. Before vs. After comparison

Paste this output into the report.

In [ ]:
def print_comparison(baseline, finetuned, view):
    rows = [
        ("Accuracy",  baseline["accuracy"],  finetuned["accuracy"]),
        ("Precision", baseline["precision"], finetuned["precision"]),
        ("Recall",    baseline["recall"],    finetuned["recall"]),
        ("F1",        baseline["f1"],        finetuned["f1"]),
    ]
    print("=" * 65)
    print(f"View: {view}")
    print("=" * 65)
    print(f"{'Metric':<12}{'Zero-shot':>18}{'Fine-tuned':>18}{'Delta':>15}")
    print("-" * 65)
    for name, b, f in rows:
        delta = f - b
        sign = "+" if delta >= 0 else ""
        print(f"{name:<12}{b:>18.3f}{f:>18.3f}{sign}{delta:>14.3f}")
    print("=" * 65)
    print(f"\nZero-shot confusion:  {baseline['confusion_matrix']}")
    print(f"Fine-tuned confusion: {finetuned['confusion_matrix']}")
    print("  (rows = true [no-tear, tear], cols = predicted [no-tear, tear])")


print_comparison(baseline_metrics, finetuned_metrics, VIEWPOINT)

In [ ]:
comparison_path = f"comparison_{VIEWPOINT}.json"
with open(comparison_path, "w") as f:
    json.dump({
        "model": model_id,
        "view": VIEWPOINT,
        "baseline": baseline_metrics,
        "finetuned": finetuned_metrics,
    }, f, indent=2)
print(f"Saved {comparison_path}")